# Day 14 · Job Task 1 — Ingest

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

This notebook is not written to be read top to bottom in class. It is written to be
**run by a job**, on a schedule, when nobody is watching.

It does three things:

| | |
|---|---|
| 1 | Pretends a new file arrived |
| 2 | Loads only what is new, using the checkpoint from the last session |
| 3 | Hands the number of new rows to the next task |

## Parameters

A job passes values in. Nothing below is hard-coded, which is the whole reason this
notebook can be run by somebody other than you.

In [ ]:
dbutils.widgets.text("my_id",         "",          "1 · Your MY_ID")
dbutils.widgets.text("catalog",       "workspace", "2 · Catalog")
dbutils.widgets.dropdown("drop_file", "yes", ["yes", "no"], "3 · Simulate a file arriving")

In [ ]:
import re, random, datetime
from pyspark.sql import functions as F

MY_ID = dbutils.widgets.get("my_id").strip()
if not MY_ID:
    raise ValueError("Parameter my_id is empty. Set it on the task, or in the widget above.")

TAG      = re.sub(r"[^a-z0-9]+", "_", MY_ID.lower()).strip("_")
CATALOG  = dbutils.widgets.get("catalog").strip() or "workspace"
SCHEMA   = f"day1213_{TAG}"

VOLUME   = "landing"
LANDING  = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/incoming"
CHK_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoints"
BRONZE   = f"{CATALOG}.{SCHEMA}.payments_bronze"

# Safe to run even if the schema was never created — a job cannot assume yesterday happened.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
dbutils.fs.mkdirs(LANDING)

print(f"schema  : {CATALOG}.{SCHEMA}")
print(f"landing : {LANDING}")

## 1 · A file arrives

In a real company an upstream system drops the file. Here the job drops it itself, so that
every scheduled run has something new to do. The file name carries the run time, so no two
runs ever produce the same name.

In [ ]:
CITIES = ["Hyderabad", "Mumbai", "Bengaluru", "Pune", "Delhi", "Chennai"]

def make_csv(seed, n_rows=100):
    rnd = random.Random(seed)
    lines = ["txn_id,city,amount,status,txn_time"]
    for i in range(n_rows):
        lines.append("{},{},{},{},{}".format(
            f"T{seed % 100000:05d}{i:04d}",
            rnd.choice(CITIES),
            rnd.randrange(50, 5000),
            rnd.choice(["SUCCESS", "SUCCESS", "SUCCESS", "FAILED"]),
            datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
    return "\n".join(lines) + "\n"


if dbutils.widgets.get("drop_file") == "yes":
    stamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    name  = f"payments_auto_{stamp}.csv"
    body  = make_csv(int(stamp.replace("_", "")) % 999983)
    try:
        with open(f"{LANDING}/{name}", "w") as fh:
            fh.write(body)
    except Exception:
        dbutils.fs.put(f"{LANDING}/{name}", body, True)
    print(f"arrived : {name}   rows: 100")
else:
    print("no file dropped this run")

## 2 · Load only what is new

Exactly the Auto Loader query from the last session, unchanged. The checkpoint it wrote
then is the checkpoint it reads now — which is why this job can be re-run any number of
times without doubling anything.

In [ ]:
CHK_BRONZE = f"{CHK_ROOT}/bronze"

before = spark.table(BRONZE).count() if spark.catalog.tableExists(BRONZE) else 0

query = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "csv")
              .option("cloudFiles.schemaLocation", f"{CHK_BRONZE}/schema")
              .option("cloudFiles.inferColumnTypes", True)
              .option("header", True)
              .load(LANDING)
         .writeStream
              .option("checkpointLocation", CHK_BRONZE)
              .trigger(availableNow=True)
              .toTable(BRONZE))

query.awaitTermination()

after    = spark.table(BRONZE).count()
new_rows = after - before

print(f"rows before : {before}")
print(f"rows after  : {after}")
print(f"new rows    : {new_rows}")

## 3 · Hand the number to the next task

A task can leave a value behind for the tasks that run after it. The next task reads it by
name. This is how two steps become one pipeline instead of two unrelated jobs.

In [ ]:
dbutils.jobs.taskValues.set(key="new_rows",   value=int(new_rows))
dbutils.jobs.taskValues.set(key="total_rows", value=int(after))

print(f"passed to the next task -> new_rows={new_rows}, total_rows={after}")